# Предварительная обработка текста на Python

Основные этапа предварительной обработки текста:
- Токенизация
- Фильтрация знаков препинания
- Приведение слов к начальной форме
- Удаление стоп-слов

Рассматривается задача определения тональности отзывов на банки.

Чтобы запускать и редактировать код, сохраните копию этого ноутбука себе (Файл -> Создать копию на Диске). Свою копию вы сможете изменять и запускать.

Учебный курс "[Программирование глубоких нейронных сетей на Python](https://openedu.ru/course/urfu/PYDNN/)".

<a target="_blank" href="https://colab.research.google.com/github/sozykin/dlpython_course/blob/master/text_processing/text_preprocessing.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


In [ ]:
# В Colab необходимо установить pymorphy3
pip install pymorphy3

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Импортируем необходимые библиотеки

In [32]:
import pandas as pd
import pymorphy3
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from pathlib import Path

In [2]:
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sozyk\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\sozyk\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Загружаем набор данных

In [33]:
data_path = 'data'

In [34]:
path = Path(data_path)

if not path.exists():
    path.mkdir(parents=True, exist_ok=True)

In [35]:
!curl -s -L -o data/banks.csv "https://www.dropbox.com/scl/fi/mhiwx4plaua183bnuqt5e/train2.csv?rlkey=wv4dalbsutmzu5br9qv5ez21v&dl=1"

## Загружаем данные в память


In [4]:
banks = pd.read_csv('data/banks.csv', index_col='id');

In [5]:
banks

,text,label
id,,
0,В Альфа-Банке работает замечательная девушка -...,1
1,Оформляя рассрочку в м. Видео в меге тёплый ст...,0
2,Очень порадовала оперативность работы в банке....,1
3,Имела неосторожность оформить потреб. кредит в...,0
4,Небольшая предыстория: Нашел на сайте MDM банк...,0
...,...,...
13994,"О высокой надёжности МКБ, порядочности и добро...",1
13995,"Обслуживаюсь в офисе на Чернореченской 42а, ка...",1
13996,Попала сегодня в очень неприятную ситуацию. Ре...,1


## Токенизация текста

In [6]:
text = banks.iloc[0]['text']

In [7]:
text

'В Альфа-Банке работает замечательная девушка - Ильясова Орна, вежливая, отзывчивая, действительно участвует в запросе клиента, я приходила к ней подряд ровно три дня, каждый день она помнила время моего прихода, помогла оформить кредит в размере 1млн рублей, когда я пришла с партнером (передавать ей полученный кредит за покупаемый мною авто), специалист Ильясова Орна помогла нам вывести всю сумму в один день (а это было непросто), так что сделка состоялась и все остались довольны! Моя знакомая в конце всего добавила: Теперь я \xa0поняла, почему мы пришли в это отделение, к этой девушке..Побольше бы таких замечательных специалистов! Приобретать программу здоровье, и вообще все, что связано с Альфа-Банком, теперь буду только у нее!'

In [8]:
tokens = word_tokenize(text.lower())

In [9]:
tokens

['в',
 'альфа-банке',
 'работает',
 'замечательная',
 'девушка',
 '-',
 'ильясова',
 'орна',
 ',',
 'вежливая',
 ',',
 'отзывчивая',
 ',',
 'действительно',
 'участвует',
 'в',
 'запросе',
 'клиента',
 ',',
 'я',
 'приходила',
 'к',
 'ней',
 'подряд',
 'ровно',
 'три',
 'дня',
 ',',
 'каждый',
 'день',
 'она',
 'помнила',
 'время',
 'моего',
 'прихода',
 ',',
 'помогла',
 'оформить',
 'кредит',
 'в',
 'размере',
 '1млн',
 'рублей',
 ',',
 'когда',
 'я',
 'пришла',
 'с',
 'партнером',
 '(',
 'передавать',
 'ей',
 'полученный',
 'кредит',
 'за',
 'покупаемый',
 'мною',
 'авто',
 ')',
 ',',
 'специалист',
 'ильясова',
 'орна',
 'помогла',
 'нам',
 'вывести',
 'всю',
 'сумму',
 'в',
 'один',
 'день',
 '(',
 'а',
 'это',
 'было',
 'непросто',
 ')',
 ',',
 'так',
 'что',
 'сделка',
 'состоялась',
 'и',
 'все',
 'остались',
 'довольны',
 '!',
 'моя',
 'знакомая',
 'в',
 'конце',
 'всего',
 'добавила',
 ':',
 'теперь',
 'я',
 'поняла',
 ',',
 'почему',
 'мы',
 'пришли',
 'в',
 'это',
 'отделен

## Фильтрация знаков препинания

In [10]:
punctuation_marks = ['!', ',', '(', ')', ':', '-', '?', '.', '..', '...', '«', '»', ';', '–', '--']

In [11]:
only_words = []
for token in tokens:
    if token not in punctuation_marks:
        only_words.append(token)

In [12]:
only_words

['в',
 'альфа-банке',
 'работает',
 'замечательная',
 'девушка',
 'ильясова',
 'орна',
 'вежливая',
 'отзывчивая',
 'действительно',
 'участвует',
 'в',
 'запросе',
 'клиента',
 'я',
 'приходила',
 'к',
 'ней',
 'подряд',
 'ровно',
 'три',
 'дня',
 'каждый',
 'день',
 'она',
 'помнила',
 'время',
 'моего',
 'прихода',
 'помогла',
 'оформить',
 'кредит',
 'в',
 'размере',
 '1млн',
 'рублей',
 'когда',
 'я',
 'пришла',
 'с',
 'партнером',
 'передавать',
 'ей',
 'полученный',
 'кредит',
 'за',
 'покупаемый',
 'мною',
 'авто',
 'специалист',
 'ильясова',
 'орна',
 'помогла',
 'нам',
 'вывести',
 'всю',
 'сумму',
 'в',
 'один',
 'день',
 'а',
 'это',
 'было',
 'непросто',
 'так',
 'что',
 'сделка',
 'состоялась',
 'и',
 'все',
 'остались',
 'довольны',
 'моя',
 'знакомая',
 'в',
 'конце',
 'всего',
 'добавила',
 'теперь',
 'я',
 'поняла',
 'почему',
 'мы',
 'пришли',
 'в',
 'это',
 'отделение',
 'к',
 'этой',
 'девушке',
 'побольше',
 'бы',
 'таких',
 'замечательных',
 'специалистов',
 'при

## Приводим слова к начальной форме

In [13]:
morph = pymorphy3.MorphAnalyzer()

In [14]:
lemmas = []
for token in only_words:
    lemmas.append(morph.parse(token)[0].normal_form)

In [15]:
lemmas

['в',
 'альфа-банк',
 'работать',
 'замечательный',
 'девушка',
 'ильясова',
 'орный',
 'вежливый',
 'отзывчивый',
 'действительно',
 'участвовать',
 'в',
 'запрос',
 'клиент',
 'я',
 'приходить',
 'к',
 'она',
 'подряд',
 'ровно',
 'три',
 'день',
 'каждый',
 'день',
 'она',
 'помнить',
 'время',
 'мой',
 'приход',
 'помочь',
 'оформить',
 'кредит',
 'в',
 'размер',
 '1млн',
 'рубль',
 'когда',
 'я',
 'прийти',
 'с',
 'партнёр',
 'передавать',
 'она',
 'получить',
 'кредит',
 'за',
 'покупать',
 'я',
 'авто',
 'специалист',
 'ильясова',
 'орный',
 'помочь',
 'мы',
 'вывести',
 'весь',
 'сумма',
 'в',
 'один',
 'день',
 'а',
 'это',
 'быть',
 'непросто',
 'так',
 'что',
 'сделка',
 'состояться',
 'и',
 'всё',
 'остаться',
 'довольный',
 'мой',
 'знакомый',
 'в',
 'конец',
 'весь',
 'добавить',
 'теперь',
 'я',
 'понять',
 'почему',
 'мы',
 'прислать',
 'в',
 'это',
 'отделение',
 'к',
 'этот',
 'девушка',
 'большой',
 'бы',
 'такой',
 'замечательный',
 'специалист',
 'приобретать',
 'п

## Удаление стоп слов

In [16]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sozyk\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [17]:
stop_words = stopwords.words("russian")

In [18]:
stop_words

['и',
 'в',
 'во',
 'не',
 'что',
 'он',
 'на',
 'я',
 'с',
 'со',
 'как',
 'а',
 'то',
 'все',
 'она',
 'так',
 'его',
 'но',
 'да',
 'ты',
 'к',
 'у',
 'же',
 'вы',
 'за',
 'бы',
 'по',
 'только',
 'ее',
 'мне',
 'было',
 'вот',
 'от',
 'меня',
 'еще',
 'нет',
 'о',
 'из',
 'ему',
 'теперь',
 'когда',
 'даже',
 'ну',
 'вдруг',
 'ли',
 'если',
 'уже',
 'или',
 'ни',
 'быть',
 'был',
 'него',
 'до',
 'вас',
 'нибудь',
 'опять',
 'уж',
 'вам',
 'ведь',
 'там',
 'потом',
 'себя',
 'ничего',
 'ей',
 'может',
 'они',
 'тут',
 'где',
 'есть',
 'надо',
 'ней',
 'для',
 'мы',
 'тебя',
 'их',
 'чем',
 'была',
 'сам',
 'чтоб',
 'без',
 'будто',
 'чего',
 'раз',
 'тоже',
 'себе',
 'под',
 'будет',
 'ж',
 'тогда',
 'кто',
 'этот',
 'того',
 'потому',
 'этого',
 'какой',
 'совсем',
 'ним',
 'здесь',
 'этом',
 'один',
 'почти',
 'мой',
 'тем',
 'чтобы',
 'нее',
 'сейчас',
 'были',
 'куда',
 'зачем',
 'всех',
 'никогда',
 'можно',
 'при',
 'наконец',
 'два',
 'об',
 'другой',
 'хоть',
 'после',
 'на

In [19]:
filtered_words = []
for token in lemmas:
    if token not in stop_words:
        filtered_words.append(token)

In [20]:
filtered_words

['альфа-банк',
 'работать',
 'замечательный',
 'девушка',
 'ильясова',
 'орный',
 'вежливый',
 'отзывчивый',
 'действительно',
 'участвовать',
 'запрос',
 'клиент',
 'приходить',
 'подряд',
 'ровно',
 'день',
 'каждый',
 'день',
 'помнить',
 'время',
 'приход',
 'помочь',
 'оформить',
 'кредит',
 'размер',
 '1млн',
 'рубль',
 'прийти',
 'партнёр',
 'передавать',
 'получить',
 'кредит',
 'покупать',
 'авто',
 'специалист',
 'ильясова',
 'орный',
 'помочь',
 'вывести',
 'весь',
 'сумма',
 'день',
 'это',
 'непросто',
 'сделка',
 'состояться',
 'всё',
 'остаться',
 'довольный',
 'знакомый',
 'конец',
 'весь',
 'добавить',
 'понять',
 'почему',
 'прислать',
 'это',
 'отделение',
 'девушка',
 'большой',
 'замечательный',
 'специалист',
 'приобретать',
 'программа',
 'здоровье',
 'вообще',
 'всё',
 'связать',
 'альфа-банк']

## Функция предварительной обработки

In [25]:
def preprocess(text, stop_words, punctuation_marks, morph):
    tokens = word_tokenize(text.lower())
    preprocessed_text = []
    for token in tokens:
        if token not in punctuation_marks:
            lemma = morph.parse(token)[0].normal_form
            if lemma not in stop_words:
                preprocessed_text.append(lemma)
    return preprocessed_text

In [26]:
punctuation_marks = ['!', ',', '(', ')', ':', '-', '?', '.', '..', '...', '«', '»', ';', '–', '--']
stop_words = stopwords.words("russian")
morph = pymorphy3.MorphAnalyzer()

In [27]:
banks[0:5].apply(lambda row: preprocess(row['text'], stop_words, punctuation_marks, morph), axis=1)

id
0    [альфа-банк, работать, замечательный, девушка,...
1    [оформлять, рассрочка, м., видео, мег, тёплый,...
2    [очень, порадовать, оперативность, работа, бан...
3    [иметь, неосторожность, оформить, потреба, кре...
4    [небольшой, предыстория, найти, сайт, mdm, бан...
dtype: object

In [28]:
banks['Preprocessed_texts'] = banks.apply(lambda row: preprocess(row['text'],stop_words, punctuation_marks, morph), axis=1)

In [29]:
banks

,text,label,Preprocessed_texts
id,,,
0,В Альфа-Банке работает замечательная девушка -...,1,"[альфа-банк, работать, замечательный, девушка,..."
1,Оформляя рассрочку в м. Видео в меге тёплый ст...,0,"[оформлять, рассрочка, м., видео, мег, тёплый,..."
2,Очень порадовала оперативность работы в банке....,1,"[очень, порадовать, оперативность, работа, бан..."
3,Имела неосторожность оформить потреб. кредит в...,0,"[иметь, неосторожность, оформить, потреба, кре..."
4,Небольшая предыстория: Нашел на сайте MDM банк...,0,"[небольшой, предыстория, найти, сайт, mdm, бан..."
...,...,...,...
13994,"О высокой надёжности МКБ, порядочности и добро...",1,"[высокий, надёжность, мкб, порядочность, добро..."
13995,"Обслуживаюсь в офисе на Чернореченской 42а, ка...",1,"[обслуживаться, офис, чернореченский, 42а, физ..."
13996,Попала сегодня в очень неприятную ситуацию. Ре...,1,"[попасть, сегодня, очень, неприятный, ситуация..."
